# Phase 10 — L-CiteEval Pilot: Spectral Features for Grounded Generation

**Goal**: Determine in 1 day on Colab A100 whether spectral features of H(n) predict
statement-level grounding faithfulness on long-context document QA.

**Dataset**: L-CiteEval / HotpotQA sub-task (multi-doc QA, 8K–48K context)  
**Model**: Falcon-3-10B-Instruct (T=1.0)  
**Samples**: 100  

**Decision gate**:
- PASS: any individual AUC > 60% AND beats Semantic Entropy by ≥ 3pp → proceed to Phase 10 main
- MARGINAL: best AUC 55–60% → run FACTS Grounding before deciding
- FAIL: best AUC ≤ 55% → pivot to Plan A (separate RAG + Agentic chapters)

Note: Semantic Entropy baseline (k=10 MC samples per statement) is deferred — too expensive
for 100 × ~5 statements × 10 samples. Gate verdict uses best individual AUC vs 55% threshold.

In [ ]:
# Cell 1 — Clone repo from master + install deps + import spectral_utils
import os, sys, shutil

REPO_DIR = '/content/hallucination_detection'

if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Do NOT touch numpy/scipy — Colab's system versions are already compatible
os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, sw_var_peak_adaptive,
    segment_by_citations, FEAT_NAMES,
    load_cache, save_cache,
    zscore, boot_auc, nadler_fuse, simple_average_fusion, best_nadler_on,
    load_lciteeval, lciteeval_prompt, lciteeval_grounding_label,
)
print('spectral_utils imported OK')

In [ ]:
# Cell 2 — Config
import torch
import numpy as np

MODEL_ID   = 'tiiuae/Falcon3-10B-Instruct'
TEMP       = 1.0
MAX_NEW    = 1024
N_SAMPLES  = 100
TASK       = 'hotpotqa'
CACHE_DIR  = '/content/drive/MyDrive/hallucination_detection/cache/phase10_pilot'
EXP_NAME   = f'falcon3_10b_{TASK}_pilot'

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Config: {MODEL_ID}, T={TEMP}, MAX_NEW={MAX_NEW}, N={N_SAMPLES}')

In [ ]:
# Cell 3 — Mount Google Drive + create cache dirs
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache dir: {CACHE_DIR}')

In [ ]:
# Cell 4 — Load model
# Falcon-3-10B is not pre-quantized; load in bfloat16 without BNB
mdl, tok = load_model(MODEL_ID, quantize_4bit=False)
print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 5 — Load L-CiteEval HotpotQA data
data = load_lciteeval(TASK, n_samples=N_SAMPLES)

# Sanity check first sample
sample = data[0]
print(f'Question: {sample["question"][:120]}...')
print(f'Docs: {len(sample["docs"])} passages')
print(f'Answers: {sample["answers"]}')
if sample['docs']:
    print(f'First doc title: {sample["docs"][0]["title"]}')

In [ ]:
# Cell 6 — Inference loop with checkpointing (save every 25 samples)
import pickle
from tqdm.auto import tqdm

checkpoint_path = os.path.join(CACHE_DIR, f'{EXP_NAME}_raw.pkl')
results = []

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'rb') as f:
        results = pickle.load(f)
    print(f'Resumed from checkpoint: {len(results)} samples done.')

start_idx = len(results)

for i in tqdm(range(start_idx, len(data))):
    row    = data[i]
    prompt = lciteeval_prompt(row)  # max 15 docs × 600 chars each

    res = generate_full(mdl, tok, prompt, T=TEMP, max_new=MAX_NEW)

    results.append({'idx': i, 'row': row, 'output': res})

    if (i + 1) % 25 == 0:
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(results, f)
        print(f'  Checkpoint saved at sample {i+1}.')

with open(checkpoint_path, 'wb') as f:
    pickle.dump(results, f)
print(f'Inference complete. {len(results)} samples saved.')

In [ ]:
# Cell 7 — Unload model
del mdl, tok
free_memory()
print('Model unloaded.')

In [ ]:
# Cell 8 — Statement segmentation
#
# generate_full returns:
#   token_entropies: one float per generated token (from model.generate scores)
#   token_offsets:   char-level spans from re-tokenizing full_text
#
# Re-tokenization can produce a slightly different token count than generation
# (BPE/chat-template boundaries differ). We trim both to the same length.

all_statements = []  # {sample_idx, text, token_start, token_end, citation_ids, ents, row}
n_no_citations = 0

for r in results:
    full_text = r['output']['full_text']
    ents      = r['output']['token_entropies']
    offsets   = list(r['output']['token_offsets'])

    # Align lengths: trim to the shorter of the two
    n = min(len(ents), len(offsets))
    ents    = ents[:n]
    offsets = offsets[:n]

    segs = segment_by_citations(full_text, offsets)

    if not segs:
        n_no_citations += 1
        continue

    for seg in segs:
        seg_ents = ents[seg['token_start']:seg['token_end']]
        all_statements.append({
            'sample_idx':   r['idx'],
            'text':         seg['text'],
            'citation_ids': seg['citation_ids'],
            'ents':         seg_ents,
            'row':          r['row'],
        })

total_samples_with_cites = len(results) - n_no_citations
print(f'Samples with ≥1 citation: {total_samples_with_cites}/{len(results)} '
      f'({100*total_samples_with_cites/len(results):.1f}%)')
print(f'Total parsed statements:  {len(all_statements)}')
print(f'Mean statements/sample:   {len(all_statements)/max(1, total_samples_with_cites):.1f}')

# Contingency: if <60% of outputs have parseable citations, report and consider model switch
if total_samples_with_cites / len(results) < 0.60:
    print('\nWARNING: <60% citation rate — Falcon-3-10B may not follow citation format.')
    print('Consider switching to Qwen2.5-72B-AWQ (Phase 8 infra available).')

In [ ]:
# Cell 9 — Per-statement feature extraction
# Filter: trace length < 10 tokens (FFT minimum).

valid_statements = []
all_feats        = []
n_too_short      = 0

for seg in all_statements:
    ents = seg['ents']
    if len(ents) < 10:
        n_too_short += 1
        continue

    feats = extract_all_features(ents)
    if feats is None:
        n_too_short += 1
        continue

    # Adaptive variance peak (handles short QA traces better)
    feats['sw_var_peak_adaptive'] = sw_var_peak_adaptive(ents)

    valid_statements.append(seg)
    all_feats.append(feats)

print(f'Retained {len(valid_statements)} valid statements ')
print(f'Filtered {n_too_short} statements (trace length < 10 tokens)')
print(f'Features per statement: {list(all_feats[0].keys()) if all_feats else "N/A"}')

In [ ]:
# Cell 10 — Ground truth labels
#
# Primary label: HotpotQA supporting_facts title matching.
# A statement is GROUNDED (1) if any cited passage comes from a gold supporting document.
# Falls back to gold-answer substring check if supporting_facts is absent.

labels = [lciteeval_grounding_label(seg['citation_ids'], seg['row'])
          for seg in valid_statements]

n_grounded   = sum(labels)
n_ungrounded = len(labels) - n_grounded
print(f'Class balance: {n_grounded} grounded / {n_ungrounded} ungrounded')
print(f'Grounding rate: {100*n_grounded/len(labels):.1f}%' if labels else 'No labels.')

# Sanity: both classes must be present for AUC to be meaningful
if n_grounded == 0 or n_ungrounded == 0:
    print('\nERROR: Only one class present — check grounding label logic.')
    print('Possible issues:')
    print('  1. supporting_facts field absent or empty → fallback to answer substring')
    print('  2. Doc titles in docs do not match supporting_facts titles')
    print('  3. All citations are [1] and passage 1 happens to always be a gold doc')
    # Spot-check 3 samples
    for seg in valid_statements[:3]:
        raw = seg['row']['raw_row']
        sf  = raw.get('supporting_facts', 'MISSING')
        print(f'  Sample supporting_facts: {sf}')
        print(f'  Citation IDs: {seg["citation_ids"]}')
        print(f'  Doc titles: {[d["title"] for d in seg["row"]["docs"][:5]]}')

In [ ]:
# Cell 11 — Statement-level AUC table (individual features)
import pandas as pd

feat_keys = list(all_feats[0].keys())
labels_arr = np.array(labels)

auc_rows = []
for k in feat_keys:
    scores = np.array([f[k] for f in all_feats])
    # boot_auc handles sign orientation: returns (auc, lo, hi)
    auc, lo, hi = boot_auc(labels_arr, scores)
    if np.isnan(auc):
        continue
    # Flip sign if below chance
    if auc < 0.5:
        auc, lo, hi = boot_auc(labels_arr, -scores)
    auc_rows.append({'feature': k, 'auc': auc, 'ci_lo': lo, 'ci_hi': hi})

df_auc = pd.DataFrame(auc_rows).sort_values('auc', ascending=False).reset_index(drop=True)
print('=== Individual Feature AUC (statement-level) ===')
print(df_auc.to_string(float_format=lambda x: f'{x:.3f}'))

In [ ]:
# Cell 12 — Nadler fusion
# Exhaustive search over feature subsets, max_size=4, ρ-filter=0.75, z-score on.

feat_dict = {k: np.array([f[k] for f in all_feats]) for k in feat_keys}

best_a, best_lo, best_hi, best_subset = best_nadler_on(
    feat_dict, feat_keys, labels_arr,
    max_size=4, label='Phase10-pilot', compare_mean=True,
)

print(f'\nBest Nadler AUC: {100*best_a:.1f}% [{100*best_lo:.1f}, {100*best_hi:.1f}]')
print(f'Best subset: {" + ".join(best_subset) if best_subset else "None"}')

In [ ]:
# Cell 13 — Semantic Entropy baseline
# Deferred: k=10 MC samples per statement × ~500 valid statements = 5000 generation passes.
# This is too expensive for a 1-day pilot on a single A100.
# The pilot gate uses best individual AUC vs 55%/60% thresholds (see Cell 16).
# SE baseline will be computed for the Phase 10 main experiment.
print('Semantic Entropy baseline: DEFERRED to Phase 10 main experiment.')
print('Pilot gate uses best individual AUC thresholds (55% / 60%).')

In [ ]:
# Cell 14 — Visual diagnostic: H(n) trajectories by class
import matplotlib.pyplot as plt

grounded_idxs   = [i for i, l in enumerate(labels) if l == 1][:5]
ungrounded_idxs = [i for i, l in enumerate(labels) if l == 0][:5]

plt.figure(figsize=(12, 5))
for j, idx in enumerate(grounded_idxs):
    plt.plot(valid_statements[idx]['ents'], color='steelblue', alpha=0.5,
             label='Grounded' if j == 0 else '')
for j, idx in enumerate(ungrounded_idxs):
    plt.plot(valid_statements[idx]['ents'], color='firebrick', alpha=0.5,
             label='Ungrounded' if j == 0 else '')

plt.title('H(n) Trajectories: Grounded (blue) vs Ungrounded (red)')
plt.xlabel('Token index (local to statement)')
plt.ylabel('Entropy H(n)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CACHE_DIR, 'phase10_hn_trajectories.png'), dpi=120)
plt.show()
print('Saved: phase10_hn_trajectories.png')

In [ ]:
# Cell 15 — Spearman heatmap (Nadler decorrelation check)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

feat_matrix = np.array([[f[k] for k in feat_keys] for f in all_feats])
corr, _     = spearmanr(feat_matrix)
if corr.ndim == 0:
    corr = np.array([[float(corr)]])

plt.figure(figsize=(10, 8))
sns.heatmap(corr, xticklabels=feat_keys, yticklabels=feat_keys,
            annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Spearman Correlation')
plt.tight_layout()
plt.savefig(os.path.join(CACHE_DIR, 'phase10_spearman_heatmap.png'), dpi=120)
plt.show()
print('Saved: phase10_spearman_heatmap.png')

In [ ]:
# Cell 16 — Decision gate
best_individual_auc = df_auc['auc'].max()
best_individual_feat = df_auc.iloc[0]['feature']

print('=' * 50)
print('PHASE 10 PILOT VERDICT')
print('=' * 50)
print(f'Best individual AUC: {100*best_individual_auc:.1f}% ({best_individual_feat})')
print(f'Nadler fusion AUC:   {100*best_a:.1f}% [{100*best_lo:.1f}, {100*best_hi:.1f}]')
print(f'Best Nadler subset:  {"+".join(best_subset) if best_subset else "None"}')
print()

if best_individual_auc > 0.60:
    verdict = 'PASS'
    action  = 'Build Phase 10 main: extend to FACTS Grounding + DeepHalluBench.'
elif best_individual_auc > 0.55:
    verdict = 'MARGINAL'
    action  = 'Run FACTS Grounding (100 samples) before deciding.'
else:
    verdict = 'FAIL'
    action  = 'Pivot to Plan A: RAG and Agentic as separate chapters, no unified Phase 10.'

print(f'VERDICT: {verdict}')
print(f'ACTION:  {action}')

In [ ]:
# Cell 17 — Save results
final_results = {
    'config': {
        'model':    MODEL_ID,
        'task':     TASK,
        'n_samples': N_SAMPLES,
        'temp':     TEMP,
        'max_new':  MAX_NEW,
    },
    'citation_rate': total_samples_with_cites / len(results),
    'n_valid_statements': len(valid_statements),
    'class_balance': {'grounded': n_grounded, 'ungrounded': n_ungrounded},
    'auc_table':     df_auc.to_dict('records'),
    'nadler': {
        'auc':    best_a,
        'ci_lo':  best_lo,
        'ci_hi':  best_hi,
        'subset': best_subset,
    },
    'verdict': verdict,
    'action':  action,
}

results_path = os.path.join(CACHE_DIR, f'{EXP_NAME}_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(final_results, f)
print(f'Results saved: {results_path}')

In [ ]:
# Cell 18 — Feature AUC bar chart
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 5))
x       = np.arange(len(df_auc))
colors  = ['steelblue' if a > 0.60 else ('gold' if a > 0.55 else 'salmon')
           for a in df_auc['auc']]

bars = ax.barh(x, df_auc['auc'], color=colors, alpha=0.85)
ax.errorbar(
    df_auc['auc'], x,
    xerr=[df_auc['auc'] - df_auc['ci_lo'], df_auc['ci_hi'] - df_auc['auc']],
    fmt='none', color='gray', capsize=3,
)
ax.axvline(0.60, color='steelblue', linestyle='--', linewidth=1, label='Pass (60%)')
ax.axvline(0.55, color='gold',      linestyle='--', linewidth=1, label='Marginal (55%)')
ax.axvline(0.50, color='red',       linestyle=':',  linewidth=1, label='Chance (50%)')
ax.set_yticks(x)
ax.set_yticklabels(df_auc['feature'])
ax.set_xlabel('AUC')
ax.set_title(f'Phase 10 Pilot — Individual Feature AUC (N={len(valid_statements)} statements)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(CACHE_DIR, 'phase10_feature_auc_bar.png'), dpi=120)
plt.show()
print('Saved: phase10_feature_auc_bar.png')